# Lesson 4 — Context and the Wall

More letters of memory make better text — and blow up the table. This
notebook builds n-gram models for n = 1 to 4 and measures the wall.

In [ ]:
# The corpus: Alice in Wonderland (public domain), with a built-in backup.
import urllib.request, re

FALLBACK = ("the small machine counted every letter of the paragraph and then began "
    "to write its own strange sentences about the city and the lake and the long "
    "quiet train ride home it wrote about the coach and the counselor and the "
    "quiet gym at seven in the morning and although every line was gibberish the "
    "shape of the words was english because the counts had come from english ") * 8

try:
    raw = urllib.request.urlopen("https://www.gutenberg.org/files/11/11-0.txt", timeout=15).read().decode("utf-8")
    raw = raw[raw.find("Alice was beginning"):raw.find("THE END")]
    print("Loaded Alice in Wonderland:", len(raw), "characters")
except Exception as e:
    raw = FALLBACK
    print("Download failed (%s) - using the built-in backup corpus." % type(e).__name__)

# Keep only lowercase letters and spaces - 27 symbols total.
corpus = re.sub(r"[^a-z ]+", " ", raw.lower())
corpus = re.sub(r" +", " ", corpus).strip()
print("Cleaned corpus:", len(corpus), "characters")
print(repr(corpus[:100]))

In [ ]:
def build(k):
    """Count what follows each k-letter context."""
    table = {}
    for i in range(len(corpus) - k):
        ctx, nxt = corpus[i:i+k], corpus[i+k]
        table.setdefault(ctx, {}).setdefault(nxt, 0)
        table[ctx][nxt] += 1
    return table

tables = {k: build(k) for k in [1, 2, 3, 4]}
for k, t in tables.items():
    print(f"context {k}: {len(t):>7,} rows seen of {27**k:>12,} possible "
          f"({100*len(t)/27**k:.4f}% coverage)")

Watch the coverage column collapse. That collapse IS the wall: almost
every long context has never occurred, so counting has nothing to say
about it.

In [ ]:
import random

def generate(k, length=150):
    table = tables[k]
    cur = corpus[:k]
    out = cur
    for _ in range(length):
        row = table.get(cur)
        if not row:
            out += " "
            cur = out[-k:]
            continue
        letters = list(row.keys())
        weights = list(row.values())
        nxt = random.choices(letters, weights=weights)[0]
        out += nxt
        cur = out[-k:]
    return out

for k in [1, 2, 3, 4]:
    print(f"--- {k} letter(s) of context")
    print(generate(k))
    print()

## Turn-in

Fill the table: for each context length — possible rows, rows seen, your
favorite generated line. Then two sentences: what improved, and what is
about to become impossible?

In [ ]:
report = """
k=1: possible=27           seen=___   favorite line:
k=2: possible=729          seen=___   favorite line:
k=3: possible=19,683       seen=___   favorite line:
k=4: possible=531,441      seen=___   favorite line:

What improved:
What is about to become impossible, and why:
"""
print(report)